# Topic Modelling with LDA

**Computational Text Analysis Pack · Module 08 — Topic Modelling**

*Technique:* Latent Dirichlet Allocation; documents as mixtures of topics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/08a_topic_modelling_lda.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Topic modelling is the most popular method in digital humanities and the most
often misused. This module teaches it and teaches the scepticism at the same
time, because the two are not separable.

**What LDA actually assumes:** every document is a mixture of a fixed number of
topics, and every topic is a distribution over words. Nothing more. The model has
no concept of meaning, sentences, or the order words come in.

What comes out is groups of words that co-occur. Whether a group is a *theme* is
your judgement, made afterwards — the model cannot supply it.

## The tooling

**scikit-learn**'s `LatentDirichletAllocation` and `NMF`. **gensim** is faster
for large corpora and **MALLET** (Java) is what most published DH topic-model
work actually used — worth knowing when you read the literature.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

LANGUAGE = "English"        # ← change this
N_TOPICS = 8                # ← and this

sub = df[df["language"] == LANGUAGE].reset_index(drop=True)

# LDA is a generative model of counts, so it takes counts, not TF-IDF.
# max_df drops words in most documents — LDA has no IDF to do it for you.
vec = CountVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=5, max_df=0.5,
                      max_features=3000)
X = vec.fit_transform(sub["text"])
vocab = np.array(vec.get_feature_names_out())
print(f"{X.shape[0]:,} documents × {X.shape[1]:,} words")

In [ ]:
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=0,
                                learning_method="batch", max_iter=25)
doc_topic = lda.fit_transform(X)
print(f"fitted {N_TOPICS} topics")
print()
for t, comp in enumerate(lda.components_):
    words = vocab[np.argsort(comp)[::-1][:12]]
    share = doc_topic[:, t].mean()
    print(f"topic {t}  ({share:>5.1%} of the corpus)  {', '.join(words)}")

## Naming topics: where the discipline goes wrong

You will now be tempted to give each list a name. Notice what happens when you
do: you are supplying the meaning, and the model gets the credit.

Two rules that keep this honest.

1. **Name a topic only after reading documents that score highly on it.**
2. **Publish the word lists, not only the names.** A reader must be able to
   disagree with your naming.

In [ ]:
TOPIC = 0                   # ← change this

top_docs = np.argsort(doc_topic[:, TOPIC])[::-1][:5]
print(f"words in topic {TOPIC}: "
      f"{', '.join(vocab[np.argsort(lda.components_[TOPIC])[::-1][:12]])}")
print()
for rank, i in enumerate(top_docs, 1):
    r = sub.iloc[i]
    print(f"{rank}. [{doc_topic[i, TOPIC]:.2f}] ({r['category']}) "
          f"{str(r['headline'])[:80]}")
    print(f"    {str(r['text'])[:220].replace(chr(10), ' ')}…")
    print()

In [ ]:
# Documents are MIXTURES. This is what distinguishes LDA from clustering,
# and it is usually thrown away by reporting only the top topic.
i = 0
print(f"headline: {sub.iloc[i]['headline'][:90]}")
print(f"category: {sub.iloc[i]['category']}")
print()
for t in np.argsort(doc_topic[i])[::-1][:4]:
    bar = "█" * int(doc_topic[i, t] * 40)
    print(f"  topic {t}: {doc_topic[i, t]:.2f} {bar}")
print()
print("Reducing this to 'the document's topic' discards the model's main claim.")

In [ ]:
# The validity check: do the topics track the known categories?
sub["top_topic"] = doc_topic.argmax(axis=1)
ct = pd.crosstab(sub["top_topic"], sub["category"])
print(ct.to_string())
print()
pct = ct.div(ct.sum(axis=1), axis=0) * 100
print("row % — a topic concentrated in one category is interpretable;")
print("a topic spread evenly is either cross-cutting or noise.")
print(pct.to_string(float_format=lambda v: f"{v:.0f}"))

### 🔧 Your turn 1

Set `N_TOPICS` to 4, then to 20, and re-run.

At 4 the topics are broad and bland; at 20 several will be near-duplicates and
some will be pure noise. There is no correct value, which is the subject of 08c.

### 🔧 Your turn 2

Change `random_state` to 1, 2, 3 and re-run at the same `N_TOPICS`.

The topics change. Not completely, but enough that the fifth-ranked word list is
different every time. Report the seed, and never build an argument on a topic
that does not survive a change of seed.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** More topics gives finer distinctions and more junk; fewer gives
stable, uninformative blends. The number is a research decision, not a parameter
to optimise — the standard approach is to fit several, read them, and choose the
one that best supports the argument you can defend. Say that plainly in your
methods rather than implying the model chose.

**Your turn 2.** LDA is stochastic. Topics are stable in broad outline and
unstable in detail, which is exactly the level at which people quote them. The
defensible practice: fit with several seeds, keep the topics that recur across
all of them, and treat the rest as an artefact of one fit. Very little published
DH topic-model work does this, which is worth knowing when you read it.

</details>

## Where this stops

You can fit and read a topic model. Next: **08b — NMF**, which does a similar job
deterministically and often more crisply.